9. Ground thruth exploration vs OSM
This notebooks explores the differences beteen the ground truth dats collected from official data repositories and the binary mask from OSM

In [ ]:
"""Collect metrics.csv and hparams.yaml from every
<root>/s{seed}/models/trained_models/version_{n}/ folder into one table."""
import os 
import sys
from pathlib import Path
 
import pandas as pd
import yaml

seed = 42 
# ---- CONFIG: adapt these to your logging keys ---------------------------------
 
 # Add the root directory to Python path so that we can import the modules
ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.append(ROOT)
print(f"ROOT: {ROOT}")
GLOB_tr = "s*/models/trained_models/version_*"
GLOB_ts = "s*/models/test_result/version_*"
 
# hparams.yaml keys (first one found wins; dotted paths reach into nested dicts)
HPARAM_KEYS = {
    "Input features": ["input_features", "features", "bands", "model.input_features"],
    "Loss function": ["loss_function", "loss", "criterion", "model.loss"],
}
feat_versions = {"version_0": "S", "version_1": "BM", "version_2": "SDT", "version_3": "S", "version_4": "BM", "version_5": "SDT", "version_6": "S", "version_7": "BM",
           "version_8": "SDT", "version_9": "S", "version_10": "BM", "version_11": "SDT", "version_12": "S", "version_13": "BM", "version_14": "SDT"}
 
# metrics.csv columns (first one found wins)
METRIC_KEYS = {
    "IoU": ["IoU", "iou"],
    "Precision": ["Precision",  "precision"],
    "Recall": ["Recall",  "recall"],
    "F1 score": ["F1 Score", "f1_score", "F1", "f1"],
    "Accuracy": ["Accuracy"],
}
# -------------------------------------------------------------------------------


def dig(d, dotted):
    """Look up 'a.b.c' in a nested dict; return None if missing."""
    for part in dotted.split("."):
        if not isinstance(d, dict) or part not in d:
            return None
        d = d[part]
    return d
 
 
def first_hit(d, keys):
    for k in keys:
        v = dig(d, k)
        if v is not None:
            return ", ".join(map(str, v)) if isinstance(v, (list, tuple)) else v
    return None
 
 
def last_valid(df, keys):
    """Last non-NaN value of the first matching column (Lightning CSV logs are sparse)."""
    for k in keys:
        if k in df.columns and df[k].notna().any():
            return df[k].dropna().iloc[-1]
    return None
 
 
def natural_key(p: Path):
    seed = p.parents[2].name  # s{seed}
    ver = p.name.split("_")[-1]
    s = int(seed[1:]) if seed[1:].isdigit() else 0
    v = int(ver) if ver.isdigit() else 0
    return s, v
 
 
rows_tr = []
for vdir in sorted(ROOT.glob(GLOB_tr), key=natural_key):
    hp_file = vdir / "hparams.yaml"
    if not hp_file.exists():
        print(f"skipping {vdir} (missing hparams.yaml)")
        continue
 
    hparams = yaml.safe_load(hp_file.read_text()) or {}
    
 
    row = {"Experiment": f"{vdir.parents[2].name}/{vdir.name}"}
    for col, keys in HPARAM_KEYS.items():
        row[col] = first_hit(hparams, keys)
    rows_tr.append(row)

# == get the test metrics from the test_result folders (if they exist) ==
rows_ts = []
for vdir in sorted(ROOT.glob(GLOB_ts), key=natural_key):
    m_file = vdir / "test_metrics.csv"
    if not (m_file.exists()):
        print(f"skipping {vdir}  (metrics.csv)")
        continue
 
    metrics = pd.read_csv(m_file)
 
    row = {"Experiment": f"{vdir.parents[2].name}/{vdir.name}","Input features": feat_versions.get(vdir.name, "Unknown")}
 
    for col, keys in METRIC_KEYS.items():
        row[col] = last_valid(metrics, keys)
    rows_ts.append(row)

# concatenate the training and test results, preferring test metrics if available
rows = {r["Experiment"]: r for r in rows_tr}
for r in rows_ts:
    if r["Experiment"] in rows:
        rows[r["Experiment"]].update(r)
    else:
        rows[r["Experiment"]] = r
table = pd.DataFrame(list(rows.values()))
table['Seed'] = table['Experiment'].apply(lambda x: int(x.split('/')[0][1:]) if x.split('/')[0][1:].isdigit() else 0)
table['Version'] = table['Experiment'].apply(lambda x: int(x.split('/')[1].split('_')[-1]) if x.split('/')[1].split('_')[-1].isdigit() else 0)
if table.empty:
    sys.exit(f"No runs found under {ROOT.resolve()} with pattern {GLOB}")
 
metric_cols = list(METRIC_KEYS)
table[metric_cols] = table[metric_cols].astype(float).round(4)

print(table.sort_values(by=["Seed","IoU","F1 score"], ascending=False).to_string(index=False))
table.sort_values(by="F1 score", ascending=False).to_csv(ROOT / f"overview_results_table{seed}.csv", index=False)
print(f"\nSaved results_table.csv and results_table.xlsx in {ROOT.resolve()}")

ROOT: d:\research_line\PUGS\code_resubmission\pugs-detection
    Experiment Input features  Loss function    IoU  Precision  Recall  F1 score  Accuracy  Seed  Version
 s42/version_1             BM        jaccard 0.7794     0.9098  0.8446    0.8760    0.9632    42        1
s42/version_13             BM jaccard_2focal 0.7784     0.9099  0.8433    0.8754    0.9631    42       13
s42/version_10             BM  jaccard_focal 0.7751     0.9062  0.8427    0.8733    0.9624    42       10
s42/version_12              S jaccard_2focal 0.7739     0.8960  0.8503    0.8725    0.9618    42       12
 s42/version_9              S  jaccard_focal 0.7694     0.8962  0.8446    0.8697    0.9611    42        9
 s42/version_0              S        jaccard 0.7677     0.8835  0.8541    0.8686    0.9602    42        0
s42/version_11            SDT  jaccard_focal 0.7657     0.8801  0.8548    0.8673    0.9598    42       11
s42/version_14            SDT jaccard_2focal 0.7657     0.8783  0.8566    0.8673    0.9597 